[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C18_Computer_Vision_Course/02_classification/02_classification.ipynb)

# 02 · 图像分类（纯 numpy 从零）

目标：在真实 **optdigits**(8×8 手写数字) 上走完经典两段式管线——**HOG-ish 手工特征 → softmax 线性分类器（从零梯度下降）**，再加 **数据增强** 与一整套 **评估口径（混淆矩阵 / precision / recall / top-k）**，每步 `assert` 自测。

**路线**：
1. 加载数据 + 训练/测试划分（纪律：统计只用 train）
2. HOG-ish 特征（梯度方向直方图 + 归一化）
3. softmax 线性分类器（交叉熵 + 梯度下降，从零）
4. 数据增强（小平移）注入不变性
5. 评估：混淆矩阵 → precision/recall、top-k
6. ✏️ 练习 → 📖 答案 → 🧪 真实 optdigits 胶囊

> **本课纪律**：先划分再一切；归一化统计只在 train 上算；增强只用于 train；固定随机种子。

## 1 · 加载数据与划分

用 sklearn 自带 8×8 手写数字（即 optdigits）。无 sklearn/无网络时回退合成。
**先划分**成 train/test，之后所有「从数据学的东西」只能用 train。

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
rng = np.random.default_rng(0)

def load_digits_or_synth(n=600, seed=0):
    try:
        from sklearn.datasets import load_digits
        d = load_digits()
        X = d.images[:n].astype(float) / 16.0
        y = d.target[:n].astype(int)
        src = 'real optdigits (sklearn)'
    except Exception:
        # 合成回退: 每个类别有**与标签相关**的固定结构(不同位置的笔画),
        # 这样分类器确实能学(否则随机标签学不动, 违背"逻辑正确"). 加噪保留真实性.
        r = np.random.default_rng(seed)
        y = r.integers(0, 10, size=n)
        X = np.zeros((n, 8, 8))
        # 为每个类预设一个不同的笔画模板(位置/朝向各异)
        protos = []
        for k in range(10):
            t = np.zeros((8, 8))
            row = 1 + (k % 5); col = 1 + (k // 5) * 3
            t[row:row+2, col:col+4] = 1.0            # 横笔画(类相关位置)
            if k % 2 == 0:
                t[row:row+4, col:col+2] = 1.0        # 偶数类加竖笔画
            protos.append(t)
        for i in range(n):
            X[i] = protos[y[i]] + 0.15 * r.standard_normal((8, 8))   # 模板+噪声
        X = np.clip(X, 0, 1); src = 'synthetic fallback'
    return X, y, src

X, y, src = load_digits_or_synth(600)
print('data source:', src, '| X', X.shape, '| classes', sorted(set(y.tolist())))
# 先划分(纪律第一步)
perm = rng.permutation(len(X))
ntr = int(0.7 * len(X))
tr, te = perm[:ntr], perm[ntr:]
Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]
assert len(set(tr.tolist()) & set(te.tolist())) == 0, 'train/test 不能重叠'
print(f'train {len(Xtr)} / test {len(Xte)}（已划分，统计只用 train）')
print('✅ 数据就绪并划分')

## 2 · HOG-ish 特征：梯度方向直方图

求 Sobel 梯度 → 把 8×8 分成 2×2 的 cell（共 16 个）→ 每 cell 统计方向直方图（按幅值加权）→ L2 归一化拼接。
验证它对**整体亮度平移不变**（HOG 鲁棒性的核心）。

In [ ]:
SX = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=float); SY = SX.T

def corr2d(img, ker):
    p = ker.shape[0]//2
    w = sliding_window_view(np.pad(img, p, mode='reflect'), ker.shape)
    return np.einsum('ijuv,uv->ij', w, ker)

def hog_ish(img, cell=2, nbins=8):
    '''简化 HOG：cell 内梯度方向(无符号 0..180)直方图, 幅值加权, 每 cell L2 归一化, 拼接。'''
    gx, gy = corr2d(img, SX), corr2d(img, SY)
    mag = np.sqrt(gx**2 + gy**2)
    ang = (np.rad2deg(np.arctan2(gy, gx)) % 180)
    H, W = img.shape
    feats = []
    for i in range(0, H, cell):
        for j in range(0, W, cell):
            m = mag[i:i+cell, j:j+cell].ravel()
            a = ang[i:i+cell, j:j+cell].ravel()
            b = np.minimum((a / (180/nbins)).astype(int), nbins-1)
            h = np.zeros(nbins)
            for bi, mi in zip(b, m):
                h[bi] += mi                          # 按幅值投票
            h = h / np.sqrt((h**2).sum() + 1e-6)     # L2 归一化(消光照)
            feats.append(h)
    return np.concatenate(feats)

f0 = hog_ish(Xtr[0])
print('特征维度:', f0.shape[0], '(16 cells x 8 bins)')
assert f0.shape[0] == 16 * 8
# 亮度整体 +0.2(裁剪到[0,1]外的部分会改变, 取中间安全区)验证近似不变
bright = np.clip(Xtr[0] * 0.8 + 0.1, 0, 1)            # 对比度/亮度变换
f1 = hog_ish(bright)
cos = f0 @ f1 / (np.linalg.norm(f0)*np.linalg.norm(f1) + 1e-12)
print(f'原图 vs 亮度变换 特征余弦相似度: {cos:.3f}')
assert cos > 0.95, 'HOG 应对亮度/对比度变换近似不变'
print('✅ HOG-ish 特征提取正确，且对亮度变换鲁棒')

In [ ]:
# 把训练/测试集都转成特征(统计无需 train, 因 HOG 是逐图固定变换)
Ftr = np.array([hog_ish(im) for im in Xtr])
Fte = np.array([hog_ish(im) for im in Xte])
# 标准化: 均值/标准差只在 train 上算(纪律!)
mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-8
Ftr = (Ftr - mu) / sd
Fte = (Fte - mu) / sd                                 # test 用 train 的统计
print('特征矩阵:', Ftr.shape, Fte.shape, '| 已用 train 统计标准化')
assert Ftr.shape[1] == Fte.shape[1] == 128
print('✅ 特征标准化完成(统计仅来自 train, 无泄漏)')

## 3 · softmax 线性分类器（从零梯度下降）

`z=XW+b` → softmax → 交叉熵。关键梯度 `dz = p - onehot(y)`（优美的形式）。
用全批量梯度下降训练，验证训练后 train 准确率显著高于随机(0.1)。

In [ ]:
def softmax(Z):
    Z = Z - Z.max(1, keepdims=True)                  # 数值稳定
    e = np.exp(Z); return e / e.sum(1, keepdims=True)

def onehot(y, K=10):
    o = np.zeros((len(y), K)); o[np.arange(len(y)), y] = 1; return o

def train_softmax(X, y, K=10, lr=0.5, epochs=300, l2=1e-3, seed=0):
    r = np.random.default_rng(seed)
    d = X.shape[1]
    W = 0.01 * r.standard_normal((d, K)); b = np.zeros(K)
    Y = onehot(y, K); n = len(X)
    for _ in range(epochs):
        P = softmax(X @ W + b)
        dZ = (P - Y) / n                              # 交叉熵对 logits 的梯度
        W -= lr * (X.T @ dZ + l2 * W)
        b -= lr * dZ.sum(0)
    return W, b

def predict(X, W, b):
    return np.argmax(X @ W + b, axis=1)

W, b = train_softmax(Ftr, ytr)
acc_tr = (predict(Ftr, W, b) == ytr).mean()
acc_te = (predict(Fte, W, b) == yte).mean()
print(f'train acc {acc_tr:.3f} | test acc {acc_te:.3f} (随机基线 0.1)')
assert acc_tr > 0.7, '训练准确率应远高于随机'
assert acc_te > 0.6, '测试准确率应明显高于随机'
print('✅ softmax 线性分类器从零训练成功')

## 4 · 数据增强：小平移注入不变性

对训练图随机平移 ±1 像素（保标签的安全增强），扩充训练集。
验证：增强后的图标签不变、且与原图不完全相同（确实做了变换）。

In [ ]:
def shift_img(img, dy, dx):
    '''整数平移, 空出处补 0(背景)。'''
    out = np.zeros_like(img)
    H, W = img.shape
    ys0, ys1 = max(0, dy), min(H, H + dy)
    xs0, xs1 = max(0, dx), min(W, W + dx)
    yt0, yt1 = max(0, -dy), min(H, H - dy)
    xt0, xt1 = max(0, -dx), min(W, W - dx)
    out[ys0:ys1, xs0:xs1] = img[yt0:yt1, xt0:xt1]
    return out

def augment(X, y, factor=1, seed=0):
    '''每张图额外生成 factor 个随机 ±1 平移版本(标签不变)。'''
    r = np.random.default_rng(seed)
    augX, augY = [X], [y]
    for _ in range(factor):
        shifted = np.array([shift_img(im, int(r.integers(-1,2)), int(r.integers(-1,2))) for im in X])
        augX.append(shifted); augY.append(y)
    return np.concatenate(augX), np.concatenate(augY)

Xa, ya = augment(Xtr, ytr, factor=1, seed=1)
print(f'增强前 {len(Xtr)} -> 增强后 {len(Xa)} 张')
assert len(Xa) == 2 * len(Xtr) and len(ya) == len(Xa)
assert np.array_equal(ya[:len(ytr)], ytr), '增强保标签'
# 平移后至少有些图变了
diff = sum(not np.array_equal(Xa[len(Xtr)+i], Xtr[i]) for i in range(50))
assert diff > 0, '增强应确实改变了部分图像'
print('✅ 数据增强：扩充一倍、标签不变、确实做了平移')

In [ ]:
# 增强训练: 在增强数据上重训, 看测试准确率(通常持平或略升; 至少不崩)
Fa = np.array([hog_ish(im) for im in Xa])
Fa = (Fa - mu) / sd
Wa, ba = train_softmax(Fa, ya)
acc_te_aug = (predict(Fte, Wa, ba) == yte).mean()
print(f'增强后 test acc {acc_te_aug:.3f} (vs 原 {acc_te:.3f})')
assert acc_te_aug > 0.55, '增强后测试准确率应仍然合理'
print('✅ 增强训练完成(测试准确率保持在合理区间)')

## 5 · 评估：混淆矩阵 / precision / recall / top-k

混淆矩阵 `C[i,j]`=真 i 预测 j。对角线=对，非对角揭示混淆。
precision(按列)、recall(按行)。top-k：真标签在概率前 k 即对。全部从零实现并对拍。

In [ ]:
def confusion_matrix(y_true, y_pred, K=10):
    C = np.zeros((K, K), dtype=int)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
    return C

def precision_recall(C):
    tp = np.diag(C).astype(float)
    prec = tp / (C.sum(0) + 1e-12)                    # 按列: 预测为 i 的里对的
    rec  = tp / (C.sum(1) + 1e-12)                    # 按行: 真为 i 的里找到的
    return prec, rec

def topk_accuracy(scores, y_true, k=3):
    topk = np.argsort(-scores, axis=1)[:, :k]         # 每行分数最高的 k 个类
    return np.mean([yt in row for yt, row in zip(y_true, topk)])

pred_te = predict(Fte, W, b)
C = confusion_matrix(yte, pred_te)
prec, rec = precision_recall(C)
# 对拍: 混淆矩阵对角线之和 / 总数 == 准确率
assert C.sum() == len(yte)
assert abs(np.diag(C).sum() / C.sum() - acc_te) < 1e-9, '混淆矩阵对角线应对应准确率'
scores_te = Fte @ W + b
top1 = topk_accuracy(scores_te, yte, k=1)
top3 = topk_accuracy(scores_te, yte, k=3)
assert abs(top1 - acc_te) < 1e-9, 'top-1 应等于普通准确率'
assert top3 >= top1, 'top-3 不应低于 top-1'
print(f'混淆矩阵对角和={np.diag(C).sum()}/{C.sum()} | macro-precision={prec.mean():.3f} macro-recall={rec.mean():.3f}')
print(f'top-1={top1:.3f}  top-3={top3:.3f}')
print('✅ 混淆矩阵/precision/recall/top-k 全部与准确率自洽')

---
## ✏️ 练习 1：cell 方向直方图

实现 `cell_histogram(mag, ang_deg, nbins=8)`：给一个 cell 的幅值与方向(度, 0..180)，返回长度 `nbins` 的**幅值加权**方向直方图（不归一化）。

In [ ]:
def cell_histogram(mag, ang_deg, nbins=8):
    # TODO: bin = min(int(ang/(180/nbins)), nbins-1); h[bin] += mag; 返回 h
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
mag = np.array([1.0, 2.0, 3.0])
ang = np.array([10.0, 100.0, 100.0])   # bin: 0, 4, 4  (180/8=22.5/bin)
h = cell_histogram(mag, ang, nbins=8)
assert h.shape == (8,)
assert abs(h.sum() - 6.0) < 1e-9, '总权重 = 幅值之和'
assert abs(h[0] - 1.0) < 1e-9 and abs(h[4] - 5.0) < 1e-9, '正确投到对应 bin'
print('✅ 练习 1 通过：方向直方图(幅值加权)正确')

## ✏️ 练习 2：softmax 交叉熵的梯度

实现 `softmax_grad(X, W, b, y)`：返回交叉熵损失对 `W` 的梯度 `dW`（含 1/n，不含正则）。
核心：`P=softmax(XW+b)`，`dZ=(P-onehot(y))/n`，`dW=X.T @ dZ`。

In [ ]:
def softmax_grad(X, W, b, y, K=10):
    # TODO: P=softmax(X@W+b); dZ=(P-onehot(y,K))/len(X); return X.T@dZ
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r = np.random.default_rng(3)
Xs = r.standard_normal((20, 5)); Ws = r.standard_normal((5, 10)); bs = np.zeros(10)
ys = r.integers(0, 10, 20)
dW = softmax_grad(Xs, Ws, bs, ys)
assert dW.shape == Ws.shape
# 数值梯度检验
def loss(W):
    P = softmax(Xs @ W + bs); return -np.log(P[np.arange(20), ys] + 1e-12).mean()
eps = 1e-5; i, j = 2, 7
Wp = Ws.copy(); Wp[i,j] += eps; Wm = Ws.copy(); Wm[i,j] -= eps
num = (loss(Wp) - loss(Wm)) / (2*eps)
assert abs(num - dW[i,j]) < 1e-4, f'解析梯度应与数值梯度一致: {dW[i,j]:.5f} vs {num:.5f}'
print('✅ 练习 2 通过：softmax 梯度与数值梯度一致')

## ✏️ 练习 3：平移增强

实现 `shift_right(img)`：把图像整体右移 1 像素（最左列补 0），其余列右移。
这是数据增强里最基本的保标签变换之一。

In [ ]:
def shift_right(img):
    # TODO: out=zeros_like(img); out[:, 1:] = img[:, :-1]; return out
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
im = np.arange(9).reshape(3, 3).astype(float)
s = shift_right(im)
assert s.shape == im.shape
assert (s[:, 0] == 0).all(), '最左列应补 0'
assert np.array_equal(s[:, 1:], im[:, :-1]), '其余列右移一格'
print('✅ 练习 3 通过：右移增强正确(保标签)')

## ✏️ 练习 4：混淆矩阵与召回率

实现 `my_confusion(y_true, y_pred, K)` 返回混淆矩阵，并 `recall_per_class(C)` 返回每类召回率。
召回率 = 对角线 / 该行之和（真为 i 的样本里被找出的比例）。

In [ ]:
def my_confusion(y_true, y_pred, K=3):
    # TODO: C=zeros((K,K),int); 对每对(t,p): C[t,p]+=1; return C
    raise NotImplementedError

def recall_per_class(C):
    # TODO: return diag(C) / (C.sum(axis=1) + 1e-12)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
yt = np.array([0, 0, 1, 1, 2, 2])
yp = np.array([0, 1, 1, 1, 2, 0])   # 类0:1对1错; 类1:2对; 类2:1对1错
C = my_confusion(yt, yp, K=3)
assert C.shape == (3, 3) and C.sum() == 6
assert C[0,0] == 1 and C[0,1] == 1 and C[1,1] == 2
rec = recall_per_class(C)
assert abs(rec[0]-0.5) < 1e-9 and abs(rec[1]-1.0) < 1e-9 and abs(rec[2]-0.5) < 1e-9
print('每类召回率:', np.round(rec, 3))
print('✅ 练习 4 通过：混淆矩阵与召回率正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cell_histogram(mag, ang_deg, nbins=8):
    h = np.zeros(nbins)
    b = np.minimum((ang_deg / (180/nbins)).astype(int), nbins-1)
    for bi, mi in zip(b, mag):
        h[bi] += mi
    return h

In [ ]:
# 练习 2 参考答案
def softmax_grad(X, W, b, y, K=10):
    P = softmax(X @ W + b)
    dZ = (P - onehot(y, K)) / len(X)
    return X.T @ dZ

In [ ]:
# 练习 3 参考答案
def shift_right(img):
    out = np.zeros_like(img)
    out[:, 1:] = img[:, :-1]
    return out

In [ ]:
# 练习 4 参考答案
def my_confusion(y_true, y_pred, K=3):
    C = np.zeros((K, K), dtype=int)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
    return C

def recall_per_class(C):
    return np.diag(C).astype(float) / (C.sum(axis=1) + 1e-12)

---
## 🧪 真实数据胶囊：optdigits 全管线 + 每类召回

在真实 optdigits 上跑完整管线（HOG → softmax → 评估），并找出**最易混的数字对**——这是评测科学家拿到结果后的第一件事：不看单一准确率，看混淆模式。

In [ ]:
# 复用前面已训练的 W,b 与测试特征 Fte
C = confusion_matrix(yte, predict(Fte, W, b))
prec, rec = precision_recall(C)
print('每类召回率:', np.round(rec, 2))
# 找最易混的 (真类, 错预测类)
off = C.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f'最易混: 真实 {i} 被错认成 {j} 共 {off[i,j]} 次')
assert C.sum() == len(yte)
assert (rec >= 0).all() and (rec <= 1).all(), '召回率应在 [0,1]'
assert off[i, j] >= 0
print('✅ 真实 optdigits 全管线跑通，并定位了最易混数字对')

**🧪 胶囊练习**：实现 `macro_f1(C)`：由混淆矩阵算每类 F1=2PR/(P+R) 再取宏平均（macro）。

In [ ]:
def macro_f1(C):
    # TODO: 由 C 算 prec,rec(参考 precision_recall); f1=2*p*r/(p+r+1e-12); 返回 f1.mean()
    raise NotImplementedError

In [ ]:
# 自测
f1 = macro_f1(C)
assert 0.0 <= f1 <= 1.0, 'macro-F1 应在 [0,1]'
assert f1 > 0.4, '合理分类器的 macro-F1 应明显 > 0'
print(f'✅ 胶囊练习通过：macro-F1 = {f1:.3f}')

In [ ]:
# 📖 胶囊参考答案
def macro_f1(C):
    p, r = precision_recall(C)
    f1 = 2 * p * r / (p + r + 1e-12)
    return float(f1.mean())

### 小结
- **管线**：划分 → HOG-ish 特征 → 标准化(统计仅 train) → softmax 分类 → 增强 → 评估。
- **HOG-ish**：梯度方向直方图 + L2 归一化，对光照/亮度变换鲁棒。
- **softmax 分类**：交叉熵梯度 `P - onehot(y)`，凸、可复现。
- **数据增强**：保标签的小平移，注入平移不变性。
- **评估**：混淆矩阵看混淆模式、precision/recall/top-k，绝不只报单一准确率。

下一站：**模块 03 · 目标检测** —— 从「整图一个标签」到「在图里定位多个框」(IoU/NMS/mAP)。